# Baseline Machine Learning Models

## CircuitBench Tutorial 4

This notebook introduces the baseline machine learning models used throughout CircuitBench. These models provide reproducible reference performance for benchmarking surrogate models used in electrical circuit design.

### Learning Objectives

- Load a benchmark dataset
- Prepare training and testing data
- Train multiple baseline models
- Compare prediction accuracy
- Measure computational efficiency
- Generate benchmark reports


## Scientific Background

Baseline models provide standardized reference points that enable fair comparison between newly proposed surrogate models and established machine learning techniques.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
import circuitbench
from circuitbench.benchmark import BenchmarkRunner

## Initialize Benchmark

In [ ]:
runner = BenchmarkRunner()

## Load Benchmark Dataset

In [ ]:
benchmark = runner.load(
    benchmark_name='CMOS_Inverter'
)

benchmark.summary()

## Prepare Features and Targets

In [ ]:
X = benchmark.X
y = benchmark.y

## Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED
)

## Baseline Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=SEED),
    'Random Forest': RandomForestRegressor(random_state=SEED),
    'KNN': KNeighborsRegressor(),
    'Support Vector Regression': SVR()
}

## Train Baseline Models

In [ ]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

print(f'Trained {len(trained_models)} baseline models.')

## Generate Predictions

In [ ]:
predictions = {}

for name, model in trained_models.items():
    predictions[name] = model.predict(X_test)

## Compute Evaluation Metrics

In [ ]:
results = []

for name, pred in predictions.items():

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

results_df = pd.DataFrame(results)
results_df

## Rank Models

In [ ]:
leaderboard = results_df.sort_values(
    by='RMSE',
    ascending=True
).reset_index(drop=True)

leaderboard.index += 1
leaderboard

## Save Leaderboard

In [ ]:
output_dir = Path('baseline_results')
output_dir.mkdir(exist_ok=True)

leaderboard.to_csv(
    output_dir/'leaderboard.csv',
    index=False
)

## Display Best Model

In [ ]:
best_model = leaderboard.iloc[0]
best_model

## Performance Comparison

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(leaderboard['Model'], leaderboard['RMSE'])
plt.xticks(rotation=30, ha='right')
plt.ylabel('RMSE')
plt.title('Baseline Model Comparison')
plt.tight_layout()

## Actual vs Predicted

In [ ]:
best_name = best_model['Model']
best_predictions = predictions[best_name]

plt.figure(figsize=(6,6))
plt.scatter(y_test, best_predictions, alpha=0.7)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted ({best_name})')

minimum = min(np.min(y_test), np.min(best_predictions))
maximum = max(np.max(y_test), np.max(best_predictions))
plt.plot([minimum, maximum], [minimum, maximum], '--')
plt.tight_layout()

## Residual Analysis

In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(8,5))
plt.hist(residuals, bins=30)
plt.xlabel('Residual')
plt.ylabel('Frequency')
plt.title('Residual Distribution')
plt.tight_layout()

## Export Predictions

In [ ]:
prediction_df = pd.DataFrame({
    'Actual': np.ravel(y_test),
    'Predicted': np.ravel(best_predictions)
})

prediction_df.to_csv(
    output_dir/'predictions.csv',
    index=False
)

## Export Benchmark Report

In [ ]:
report = {
    'best_model': str(best_name),
    'mae': float(best_model['MAE']),
    'rmse': float(best_model['RMSE']),
    'r2': float(best_model['R2']),
    'random_seed': SEED
}

import json

with open(output_dir/'benchmark_report.json', 'w') as f:
    json.dump(report, f, indent=4)


## Reproducibility Checklist

- Fixed random seed used throughout the experiment.
- Training and testing split documented.
- Evaluation metrics computed consistently.
- Predictions exported for future analysis.
- Leaderboard generated automatically.
- Benchmark report saved in JSON format.


## Best Practices

1. Always compare new models against baseline methods.
2. Report multiple evaluation metrics rather than a single score.
3. Use identical train/test splits for fair benchmarking.
4. Record software versions and random seeds.
5. Export predictions and metadata for reproducibility.


## Next Notebook

Continue with **5_Model_Evaluation.ipynb** to perform comprehensive performance evaluation, statistical significance testing, confidence interval estimation, calibration analysis, and publication-quality visualization.

# Summary

In this notebook we:

- Loaded a benchmark dataset.
- Trained multiple baseline regression models.
- Compared predictive performance using MAE, RMSE, and R².
- Ranked models on a reproducible leaderboard.
- Visualized prediction accuracy and residual distributions.
- Exported benchmark reports, predictions, and evaluation results.

These baseline models establish reference performance for future surrogate-model benchmarking within CircuitBench.